# Environment config

**Objective:** Keep deployment-specific values outside the code and validate them once at startup.

## Simple version

Read a value from the deployment environment and convert it to the type the application needs.

In [ ]:
deployment_env = {
    "APP_PORT": "8000",
    "APP_DATABASE_URL": "postgresql://localhost/app",
}

port = int(deployment_env["APP_PORT"])
database_url = deployment_env["APP_DATABASE_URL"]

print(port, database_url)

## Polished version

One settings object defines the contract shared by development, staging, and production. Environments change values—not application code.

In [ ]:
from collections.abc import Mapping
from dataclasses import dataclass


class ConfigError(ValueError):
    pass


@dataclass(frozen=True)
class Settings:
    port: int
    database_url: str
    redis_url: str
    release: str

    @classmethod
    def from_env(cls, env: Mapping[str, str]) -> "Settings":
        names = (
            "APP_PORT",
            "APP_DATABASE_URL",
            "APP_REDIS_URL",
            "APP_RELEASE",
        )
        missing = [name for name in names if not env.get(name)]
        if missing:
            raise ConfigError(f"Missing config: {', '.join(missing)}")

        port = int(env["APP_PORT"])
        if not 1 <= port <= 65_535:
            raise ConfigError("APP_PORT must be between 1 and 65535")

        return cls(
            port=port,
            database_url=env["APP_DATABASE_URL"],
            redis_url=env["APP_REDIS_URL"],
            release=env["APP_RELEASE"],
        )


development = Settings.from_env(
    {
        "APP_PORT": "8000",
        "APP_DATABASE_URL": "postgresql://localhost/app",
        "APP_REDIS_URL": "redis://localhost/0",
        "APP_RELEASE": "dev",
    }
)
production = Settings.from_env(
    {
        "APP_PORT": "8080",
        "APP_DATABASE_URL": "postgresql://db.internal/app",
        "APP_REDIS_URL": "rediss://redis.internal/0",
        "APP_RELEASE": "2026.08.31",
    }
)

print("Development:", development)
print("Production:", production)

## Applied in this repository

The [REST settings](../00P1-project-rest-api/app/config.py) and [LLM settings](../00P2-project-llm-api/app/config.py) read deployment values from the environment. Their Docker Compose services keep local backing services similar to production.